[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-gmm.ipynb)

# Gaussian Mixture Models & the EM Algorithm

*AIBits Academy · Machine Learning End To End · Unsupervised Learning · New*

K-Means forces every point into exactly one cluster with a hard boundary. GMM asks a softer, more honest question: what's the probability this point belongs to each cluster?

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Hard vs Soft Clustering

K-Means assigns each point to its single nearest centroid — a hard, all-or-nothing decision. A Gaussian Mixture Model instead assumes the data was generated by a weighted mixture of K Gaussian distributions, and computes each point's **probability of membership** in every cluster:

$$p(x) = \sum_{k=1}^{K} \pi_k \, \mathcal{N}(x \mid \mu_k, \Sigma_k) \qquad \text{where } \sum_k \pi_k = 1$$

πₖ is the mixture weight (prior probability) of cluster k, and 𝒩(x|μₖ,Σₖ) is the Gaussian density with mean μₖ and covariance Σₖ. Critically, each cluster gets its *own* covariance matrix — meaning GMM can naturally represent elongated or tilted elliptical clusters, something K-Means' pure-distance assignment cannot.

## The EM Algorithm — Fitting a GMM

The mixture weights, means, and covariances can't be solved in closed form because we don't observe which cluster generated each point — that's a hidden (latent) variable. Expectation-Maximization solves this by alternating between two steps until convergence:

- **E-step (Expectation):** Given the current parameters, compute the "responsibility" γᵢₖ — the probability that cluster k generated point i, for every point and every cluster (soft assignment)

- **M-step (Maximization):** Given the responsibilities, re-estimate πₖ, μₖ, Σₖ as the responsibility-weighted mean/covariance of all points, for each cluster

- Repeat until the log-likelihood stops improving meaningfully

$$\begin{gathered}\gamma_{ik} = \dfrac{\pi_k \mathcal{N}(x_i \mid \mu_k, \Sigma_k)}{\sum_j \pi_j \mathcal{N}(x_i \mid \mu_j, \Sigma_j)} \quad \text{(E-step)} \\[6pt] \mu_k = \dfrac{\sum_i \gamma_{ik} x_i}{\sum_i \gamma_{ik}} \quad \text{(M-step)}\end{gathered}$$

This is precisely "soft K-Means": the E-step is analogous to K-Means' assignment step (but probabilistic, not hard), and the M-step is analogous to K-Means' centroid update (but weighted by responsibility, not a hard 0/1 membership). Each EM iteration is guaranteed to never decrease the data log-likelihood — it converges, though possibly to a local optimum, exactly like K-Means' own convergence guarantee.

## Watch EM Converge — Ellipses Finding the Clusters

A real EM run on two elongated, correlated Gaussian blobs (verified against scikit-learn's `GaussianMixture`). Each iteration redraws the two fitted Gaussians as 2-sigma ellipses and recolours every point by its soft responsibility γ (blue ↔ orange blend). Starting from a deliberately off-centre, circular guess, watch the ellipses stretch, rotate, and slide into place while the log-likelihood climbs at every step — the M-step made visible.

## Code — GMM on Mumbai Customer Spending, Compared to K-Means

In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans

# Customer segments with genuinely different (elongated) spending covariance shapes
np.random.seed(6)
budget    = np.random.multivariate_normal([20,8],  [[15,8],[8,6]],   150)  # correlated spread
premium   = np.random.multivariate_normal([70,25], [[40,-15],[-15,20]], 100)  # negatively correlated
occasional= np.random.multivariate_normal([35,40], [[10,0],[0,35]],   120)
X = np.vstack([budget, premium, occasional])

gmm = GaussianMixture(n_components=3, covariance_type='full', random_state=42).fit(X)
km  = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)

# GMM gives a full probability distribution over clusters per point, not just a label
sample_point = X[0:1]
probs = gmm.predict_proba(sample_point)
print(f"GMM soft membership for one point: {np.round(probs[0], 3)}")
print(f"K-Means hard label for same point:  cluster {km.predict(sample_point)[0]}")
print(f"\nGMM   BIC = {gmm.bic(X):.1f}   (lower is better, penalises complexity)")
print(f"GMM converged in {gmm.n_iter_} EM iterations")

The point is 89.1% likely to belong to cluster 0, but genuinely has an 8.8% chance of belonging to cluster 2 — a customer near a segment boundary. K-Means can only ever report "cluster 0", discarding this ambiguity entirely.

## Choosing K — the Bayesian Information Criterion (BIC)

Unlike K-Means' elbow/silhouette methods, GMM has a natural model-selection tool: BIC trades off how well the mixture fits the data (log-likelihood) against model complexity (number of parameters, which grows with K and with covariance flexibility):

$$\mathrm{BIC} = -2\ln(\hat{L}) + p\ln(n) \qquad \text{where } p = \text{number of free parameters},\ n = \text{sample size}$$

In [ ]:
bics = []
for k in range(1, 8):
    g = GaussianMixture(n_components=k, covariance_type='full', random_state=42).fit(X)
    bics.append(g.bic(X))
    print(f"k={k}  BIC={g.bic(X):8.1f}")
best_k = np.argmin(bics) + 1
print(f"\nBest k by BIC: {best_k}")

## K-Means vs GMM

|  | K-Means | Gaussian Mixture Model |
|---|---|---|
| Assignment | Hard (one cluster per point) | Soft (probability distribution over clusters) |
| Cluster shape | Spherical (Euclidean distance to centroid) | Elliptical, arbitrary orientation (full covariance) |
| Model selection | Elbow method, silhouette score | BIC / AIC (principled likelihood-based criterion) |
| Underlying algorithm | Lloyd's algorithm (a special case of EM) | Expectation-Maximization |
| Density estimation | No — can't compute p(x) for a new point | Yes — full generative probability model |

## Try It — Soft Membership for a New Customer

Click anywhere on the plot to place a new Mumbai customer (Monthly Spend vs Visits/Month) and see their **true** responsibility γ across the three segments below — computed from the same three Gaussian components (with their real, correlated covariance shapes) used to generate the data above. Compare this to what a hard nearest-centroid (K-Means-style) rule would have said instead.

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Recover two hidden groups

`x` mixes two populations (means near 0 and 5). Fit `GaussianMixture(n_components=2, random_state=0)` and store the sorted component means in `means`.

In [ ]:
import numpy as np
from sklearn.mixture import GaussianMixture
rng = np.random.default_rng(0)
x = np.concatenate([rng.normal(0, 1, 300), rng.normal(5, 1, 300)]).reshape(-1, 1)
means = None   # TODO


In [ ]:
try:
    check("means near 0 and 5", np.allclose(means, [0, 5], atol=0.3))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from sklearn.mixture import GaussianMixture
rng = np.random.default_rng(0)
x = np.concatenate([rng.normal(0, 1, 300), rng.normal(5, 1, 300)]).reshape(-1, 1)
gm = GaussianMixture(n_components=2, random_state=0).fit(x)
means = np.sort(gm.means_.ravel())

```

</details>

### Exercise 2 · Medium · Soft assignments

Unlike k-means, a GMM gives probabilities. Store in `p_mid` the probabilities of the two components for a customer at x = 2.5 (halfway), and in `p_far` for x = 5.0.

In [ ]:
p_mid = p_far = None   # TODO (reuse gm from the previous exercise)


In [ ]:
try:
    check("midpoint is ambiguous", max(p_mid) < 0.8)
    check("x=5 is clearly one group", max(p_far) > 0.99)
    check("probabilities sum to 1", abs(p_mid.sum() - 1) < 1e-9)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
p_mid = gm.predict_proba([[2.5]])[0]
p_far = gm.predict_proba([[5.0]])[0]

```

</details>

### Exercise 3 · Stretch · Pick the number of components with BIC

Fit mixtures with 1–5 components, compute the BIC of each, and store the best (lowest-BIC) count in `best_k` and the list of BICs in `bics`.

In [ ]:
bics = []
best_k = None   # TODO (reuse x)


In [ ]:
try:
    check("five BIC values", len(bics) == 5)
    check("BIC picks 2 components", best_k == 2)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.mixture import GaussianMixture
bics = [GaussianMixture(n_components=k, random_state=0).fit(x).bic(x) for k in range(1, 6)]
best_k = int(np.argmin(bics)) + 1

```

BIC rewards fit but charges for every extra parameter, so it stops at the true number of groups.

</details>

---
*Back to the course: **Machine Learning End To End → Gaussian Mixture Models & the EM Algorithm**.*